# Session 6 - Validation, Model Selection & Tuning

**Block 2: Machine Learning** · 4 hours · ⭐ **the conceptual keystone of the course**

---

## Learning objectives

By the end of this session you will be able to:

1. Design a cross-validation strategy that fits the structure of the data, and
   justify it from evidence. `[CLO4]`
2. Construct a tuning experiment free of both leakage and selection bias. `[CLO3, CLO4]`
3. Attach an uncertainty interval to a metric, and decide whether two models
   genuinely differ. `[CLO4]`
4. Explain what nested cross-validation is for. `[CLO4]`
5. **Investigate an implausibly strong feature and re-scope a problem in
   response.** `[CLO1, CLO3]`

## Prerequisites

Sessions 1–5. And the question that has been on the board for three weeks.

## Why does this matter?

Today we answer **"is 0.83 real?"**

The answer is no. But the interesting part is not the answer - it is that nothing
you have learned so far could have told you, and the thing that finally reveals it
is not a statistical test. It is reading a column definition carefully.

You will also learn to stop reporting single numbers. By the end of today,
"R² = 0.606" will look to you like an incomplete sentence.

## §1 - Retrieval practice

From memory. Five minutes.

1. A do-nothing classifier scored F1 = 0.799. Why?
2. What is the floor of PR-AUC?
3. The depth-5 tree had the best AUC and we did not deploy it. Why not?
4. Ridge changed Session 4's R² by 0.0002. What did that tell us?
5. What has been written on the board for three weeks?

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "src").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
# scipy.stats for the paired t-test in §5: today we stop eyeballing differences.
from scipy import stats

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
# permutation_importance is the diagnostic that exposes the feature we have been
# quietly tolerating since Session 3.
from sklearn.inspection import permutation_importance
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score
from sklearn.model_selection import (GridSearchCV, GroupKFold, GroupShuffleSplit,
                                     KFold, RandomizedSearchCV, cross_val_score)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from src.data import SEED, load_raw, set_seed, split_by_host

sns.set_theme(style="whitegrid")
set_seed(SEED)

df = load_raw()
train, _ = split_by_host(df)
train["price_num"] = (
    train["price"].astype(str).str.replace(r"[^0-9.]", "", regex=True)
    .replace("", np.nan).astype(float)
)
work = train[train["price_num"].notna()].copy()
work = work[work["price_num"].between(10, 2000)]
y = np.log1p(work["price_num"])
groups = work["host_id"]

NUMERIC = ["accommodates", "bedrooms", "beds", "bathrooms", "latitude", "longitude",
           "minimum_nights", "number_of_reviews", "review_scores_rating",
           "calculated_host_listings_count", "hosts_time_as_host_years"]
CATEGORICAL = ["room_type", "property_type", "neighbourhood_cleansed"]


def make_preprocessor(numeric=NUMERIC, categorical=CATEGORICAL):
    # A factory rather than a single shared object. Later in the session we drop a
    # column from NUMERIC, and a preprocessor is stateful once fitted, so building a
    # fresh one per experiment avoids reusing something fitted on other columns.
    return ColumnTransformer([
        ("num", Pipeline([("impute", SimpleImputer(strategy="median")),
                          ("scale", StandardScaler())]), numeric),
        ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                          ("onehot", OneHotEncoder(handle_unknown="ignore",
                                                   min_frequency=20,
                                                   sparse_output=False))]), categorical),
    ])


print(f"modelling rows: {len(work):,}")

## §2 - Four words that get confused

| Term | What it is | Who sets it | Example |
|---|---|---|---|
| **Parameter** | learned from data by fitting | the algorithm | a regression coefficient |
| **Hyperparameter** | fixed before fitting; controls *how* fitting happens | you | Ridge's α, a tree's `max_depth`, KNN's k |
| **Training set** | data used to learn parameters | - | - |
| **Validation set** | data used to choose hyperparameters *and* to choose between models | - | - |
| **Test set** | data used **once**, to estimate future performance | - | our sealed parquet |

The distinction that matters: **a hyperparameter chosen by looking at some data is
fitted to that data**, just by a slower and less systematic optimiser - you. If you
choose α by looking at the test set, the test set has trained your model.

This is why we need three roles, not two. Parameters are fitted on train.
Hyperparameters are fitted on validation. Only then is the test set an estimate of
anything.

## §3 - Why one split is not enough

We have been using 5-fold cross-validation without justifying it. Here is the
justification.

### Predict before you run

Take the same model and the same data, and split it 80/20 by host **thirty different
times** with thirty different random seeds. Each time, fit and score.

**Write down**: how far apart do you expect the best and worst R² to be?

### The folds, drawn

![5 fold CV](../assets/diagrams/s06_validation/5-fold-CV.png)

![2 fold CV](../assets/diagrams/s06_validation/2-fold-CV.png)

Five trials as five strips, the validation block moving along one place at a time; then the two-fold case, which is the same idea at its smallest.

The histogram above says that a single split is one draw from a wide distribution. This is what we do instead: take every draw and report the spread as well as the average. Note that each row is a **complete refit** from scratch - which is why five-fold costs five times a single split, and why nothing may be fitted outside the strip it belongs to.

In [ ]:
# TODO: Prediction first.
#
#   My prediction: the best and worst of 30 single splits will differ by about ____
#
# Then run the experiment: 30 GroupShuffleSplit splits (test_size=0.2, seeds 0..29),
# fit a Ridge pipeline on each, score on the held-out part, and report min/max/sd.

### Read that again

**0.726 to 0.843.** A range of 0.117, from nothing but the choice of random seed.

So if you fit one model, split once, and report the number, you are reporting a
draw from that distribution. Nothing stops you drawing 0.843 and calling it your
result. Nothing stops a colleague drawing 0.726 from the same model and concluding
it is unusable.

> **A single held-out score is not a measurement. It is a sample of size one from a
> distribution you have not looked at.**

This is what cross-validation buys: instead of one draw, you take *k* draws that
between them use every row exactly once for validation, and you report the mean
**and the spread**.

It also tells you something uncomfortable about published results. Anyone reporting
a single split, on data like this, has an undisclosed ±0.06 of freedom in what they
announce.

## §4 - Choosing a splitter

Cross-validation is not one procedure. The right variant depends on the structure of
your data, and choosing wrongly reintroduces the leakage you fixed in Session 2.

| Splitter | Use when | Guarantees |
|---|---|---|
| `KFold(shuffle=True)` | rows are genuinely independent | every row validates once |
| `StratifiedKFold` | classification, especially imbalanced | class balance preserved per fold |
| `GroupKFold` | rows cluster into entities | no group spans train and validation |
| `TimeSeriesSplit` | order matters | never trains on the future |

We established in Session 2 that 80.5% of our listings share a host with another
listing. Let us measure what ignoring that costs - inside cross-validation this
time, not just in the initial split.

### The loop the folds sit inside

![CV](../assets/diagrams/s06_validation/CV.png)

Fit on one part, evaluate on the part held out, repeat, average.

Grouping changes nothing in this picture except **which rows are allowed to share a strip**. That is the entire content of `GroupKFold`: the machinery is identical, and the only new rule is that a host's listings travel together.

In [ ]:
# What grouping is worth, measured. One model, one dataset, two ways of building
# folds, and the difference between them is the size of the lie a random split tells.
pipe_gb = Pipeline([("prep", make_preprocessor()),
                    ("model", HistGradientBoostingRegressor(random_state=SEED))])

# Plain KFold ignores host entirely, so a host's listings land on both sides of each
# fold boundary and the model can recognise its neighbours rather than generalise.
random_kf = cross_val_score(pipe_gb, work[NUMERIC + CATEGORICAL], y,
                            cv=KFold(5, shuffle=True, random_state=SEED), scoring="r2")
# GroupKFold keeps every host whole, so each fold is a genuinely unseen set of hosts.
grouped_kf = cross_val_score(pipe_gb, work[NUMERIC + CATEGORICAL], y,
                             cv=GroupKFold(5), groups=groups, scoring="r2")

print(f"  random  KFold : R² = {random_kf.mean():.4f}  (sd {random_kf.std():.4f})")
print(f"  GroupKFold    : R² = {grouped_kf.mean():.4f}  (sd {grouped_kf.std():.4f})")
# This gap is free score, and it is not real. Nobody would ever detect it from the
# random-KFold numbers alone, which look perfectly respectable on their own.
print(f"  inflation     : {random_kf.mean() - grouped_kf.mean():+.4f}")
# The grouped folds are also more variable, and that is honest rather than worse:
# hosts differ, so folds of different hosts genuinely differ.
print(f"\n  per-fold, random  : {np.round(random_kf, 4)}")
print(f"  per-fold, grouped : {np.round(grouped_kf, 4)}")

### Two things, and the second is the nastier one

**The score is inflated by +0.045.** Expected - the same operator's listings appear
on both sides, so the model is partly being tested on what it memorised.

**The wrong method also looks more reliable.** Random KFold's fold-to-fold standard
deviation is **0.003**; GroupKFold's is **0.013** - more than four times wider. On
the classification task the gap is starker still: 0.004 against 0.039, a factor of
ten. Look at the per-fold lists above: the random folds agree to three decimal
places.

Think about what that means. If you ran both and had to guess which was
trustworthy, the tight, consistent, high-scoring one looks like the careful
analysis. It is the broken one. Its folds agree with each other because they are all
making the same mistake.

> **A narrow confidence interval is not evidence that an estimate is correct.** It
> only says the estimate is stable, and a systematically wrong procedure is very
> stable indeed.

## §5 - The investigation

We are at R² ≈ 0.83 with three leaks removed. The question has been on the board for
three weeks.

Start where you always start: **what is the model actually using?**

In [ ]:
# One split, fitted once, so the importance can be measured on data the model has
# not seen. Permutation importance on training data would reward memorisation.
tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=SEED)
              .split(work, y, groups=groups))
audit_model = Pipeline([("prep", make_preprocessor()),
                        ("model", HistGradientBoostingRegressor(random_state=SEED))])
audit_model.fit(work.iloc[tr][NUMERIC + CATEGORICAL], y.iloc[tr])

# Permutation importance shuffles one column at a time and measures how far R² falls.
# It is model-agnostic and measured on held-out data, which is why it is trusted here
# over a tree's built-in impurity importance.
importance = permutation_importance(
    # n_repeats=5 because each shuffle is itself random; one shuffle is one draw.
    audit_model, work.iloc[te][NUMERIC + CATEGORICAL], y.iloc[te],
    n_repeats=5, random_state=SEED, scoring="r2")
ranked = pd.Series(importance.importances_mean,
                   index=NUMERIC + CATEGORICAL).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 4.2))
# iloc[::-1] reverses the order so the largest bar ends up at the top of a barh.
ranked.head(8).iloc[::-1].plot.barh(ax=ax, color="indianred")
ax.set(title="permutation importance - drop in R² when the column is shuffled",
       xlabel="ΔR²")
plt.tight_layout()
plt.show()
# Read the top of this list and ask whether it makes sense. A booking constraint
# should not out-predict capacity and location when the question is nightly price.
print(ranked.head(6).round(4).to_string())

### One feature is worth five of everything else

`minimum_nights`: **0.646**. The next feature, `room_type`, is 0.137.

Shuffling one column costs us nearly two thirds of our R². For a *pricing* model,
that is a strange thing for the minimum-stay policy to do. Capacity, room type and
location should dominate what a flat charges. A booking rule should not.

It is not a leak in any of the four senses from Session 3 - `minimum_nights` is
genuinely known before you set a price. So what is going on?

**Your task: find out what `price` actually measures.** Twenty minutes.

Hints, in order - use as few as possible:
1. There are four `price_quote_*` columns. We removed two of them as leaks. Look at
   the other two.
2. A price is money per night. What span of nights is our price quoted for?
3. Compare that span against `minimum_nights`.

In [ ]:
# TODO: Work out why minimum_nights is so predictive of price.
#
# Start from the price_quote_checkin_date and price_quote_checkout_date columns.

### What `price` actually is

The scraper asks Airbnb for a quote. The stay it asks about has a length driven by
the listing's own minimum-stay rule - the correlation is **0.98**. And long stays
get per-night discounts:

| minimum_nights | median quoted nights | median price |
|---|---:|---:|
| 1 night | 1 | €259 |
| 2–3 | 2 | €260 |
| 4–7 | 5 | €228 |
| 8–31 | 30 | €100 |
| 32+ | 32 | €70 |

So `price` is **not** "the nightly price of this listing". It is:

> *the per-night price quoted for a stay whose length equals this listing's minimum
> stay.*

A €70 long-stay listing and a €259 short-stay listing are not necessarily priced
differently for the same product. They are quoted on **different products**.

`minimum_nights` is not predicting the market. It is telling the model **which
pricing regime the scraper sampled**, and once the model knows the regime it has
most of the answer.

### Is this leakage?

This is the question worth the whole session. `minimum_nights` is known at
prediction time, so it passes the deployment test from Session 3. By the taxonomy
we built, it is not leakage.

And yet a model relying on it is answering a question nobody asked.

The resolution: **whether this feature is legitimate depends on the client.**

- **Client B asks:** *"what should we charge per night for this flat?"* Then the
  target must mean one consistent thing. Mixing one-night quotes with 32-night
  quotes means the model is partly predicting *our own sampling procedure*. It is
  invalid.
- **A revenue analyst asks:** *"what per-night revenue does a listing of this type
  realise?"* Then the long-stay discount is a real feature of the market and
  belongs in the model.

> **Leakage is not a property of a column. It is a relationship between a column,
> a target, and a decision.** No amount of staring at correlations would have told
> you this. You had to read what the data *means*.

### Re-scope, and pay the price

We serve Client B. So we restrict to comparable short-stay listings and drop the
regime indicator.

In [ ]:
# TODO: Re-scope the problem for Client B.
#
#   1. Keep only listings with minimum_nights <= 3.
#   2. Drop minimum_nights from the feature list.
#   3. Score Ridge and HistGB with GroupKFold, using the SAME folds for both.
#   4. Report mean and standard deviation for each.
#
# How many rows did you lose? What happened to R²?

### The full descent

| Stage | R² | What we learned |
|---|---:|---|
| every numeric column | 0.999 | `price_quote_price_per_night` *is* the target |
| minus the identical column | 0.973 | revenue ÷ occupancy rebuilds it |
| minus revenue + occupancy | 0.957 | total_price ÷ nights rebuilds it too |
| minus total_price | 0.834 | three leaks gone; still not honest |
| **re-scoped, regime indicator dropped** | **0.606** | the target now means one thing |

**0.999 → 0.606.** Every step was an improvement in the analysis, and every step
lowered the score.

We also discarded 42% of our data to get here. That is a real cost, honestly
incurred: those 4,600 long-stay listings are a different product, and a model that
averages across both answers neither question well.

> If your instinct is that 0.606 is a worse result than 0.834, you have not yet
> absorbed what this course is for. 0.606 is the first number today that means
> anything at all.

## §6 - Uncertainty, and whether a difference is real

Look again at the two scoped results:

- Ridge: **0.606 ± 0.100**
- HistGB: **0.591 ± 0.069**

Ridge is ahead by 0.015. Report "Ridge wins" and you would be making the most
common error in applied machine learning.

### First: how wide is the fold spread?

In [ ]:
# Plot the folds, not the mean. Two models whose means differ by a hair but whose
# per-fold clouds overlap completely are not two different models, and a bar chart
# of two averages would hide exactly that.
fig, ax = plt.subplots(figsize=(9, 3.4))
for i, (name, s) in enumerate(scoped.items()):
    # One row per model, one dot per fold.
    ax.scatter(s, [i] * len(s), s=70, alpha=0.8, label=name)
    # A tall thin marker for the mean, so it reads as a summary of the dots rather
    # than as another observation.
    ax.scatter([s.mean()], [i], marker="|", s=800, color="black")
ax.set_yticks(range(len(scoped)))
ax.set_yticklabels(scoped.keys())
ax.set(title="per-fold R² - the black bar is the mean", xlabel="R²")
plt.tight_layout()
plt.show()

# The same picture as numbers: compare each model's range against the gap between
# their means. The ranges are far wider than the gap.
for name, s in scoped.items():
    print(f"  {name:8s} range {s.min():.3f} – {s.max():.3f}   mean {s.mean():.4f}")

Ridge's folds run from 0.510 to 0.743. Its own worst fold is 0.23 below its own
best. Against that, a 0.015 gap between models is noise.

### Second: the paired comparison

The right test is **not** "do the two distributions overlap". Both models were
evaluated on the *same folds*, so we compare them **fold by fold** and ask whether
the differences are consistently signed. Pairing removes the fold-difficulty
variance, which is the largest source of noise.

In [ ]:
# TODO: Compare Ridge and HistGB properly.
#
#   1. Compute the per-fold difference (HistGB - Ridge).
#   2. Report the mean difference and its standard deviation.
#   3. Run a paired t-test (scipy.stats.ttest_rel).
#   4. State your conclusion in one sentence a client could read.

**p = 0.531.** The differences are −0.091, +0.041, −0.021, +0.009, −0.012: they do
not even agree on a sign.

Compare that with Session 5's classification result, where boosting beat logistic
regression by +0.036 with the **same sign in all five folds** and p = 0.0017. That
was a real difference. This is not.

> **Two models are only different if their per-fold differences are consistently
> signed.** Comparing means, or eyeballing overlapping error bars, gives the wrong
> answer in both directions.

### Third: a confidence interval on a single estimate

Sometimes you have one held-out set and no folds. You can still quantify
uncertainty, by resampling the predictions you already have.

In [ ]:
# An interval for a single fitted model, via the bootstrap. random_state=1 rather
# than SEED so this split is independent of the audit split used in §3.
tr2, te2 = next(GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=1)
                .split(short, y_short, groups=groups_short))
final = Pipeline([("prep", make_preprocessor(NUMERIC_SHORT, CATEGORICAL)),
                  ("model", Ridge())])
final.fit(short.iloc[tr2][NUMERIC_SHORT + CATEGORICAL], y_short.iloc[tr2])

y_true = y_short.iloc[te2].to_numpy()
y_pred = final.predict(short.iloc[te2][NUMERIC_SHORT + CATEGORICAL])

rng = np.random.default_rng(SEED)
# Resample the held-out rows with replacement 2,000 times and re-score each
# resample. The model is never refitted: this measures how much the *score* depends
# on which listings happened to be in the evaluation set.
boot = np.array([
    r2_score(y_true[idx], y_pred[idx])
    for idx in (rng.integers(0, len(y_true), len(y_true)) for _ in range(2000))
])
# The middle 95% of those 2,000 scores is the percentile confidence interval.
lo, hi = np.percentile(boot, [2.5, 97.5])

print(f"point estimate      R² = {r2_score(y_true, y_pred):.4f}")
# The width is the number to quote. Any claimed improvement narrower than this is
# not distinguishable from the choice of evaluation rows.
print(f"bootstrap 95% CI       [{lo:.4f}, {hi:.4f}]   width {hi - lo:.4f}")
# Two different quantities: one fitted model on one split, against the average over
# five. They should be close, and they answer different questions.
print(f"cross-validated mean   {scoped['Ridge'].mean():.4f}")

Two things worth noticing.

**The interval is 0.13 wide.** With about 1,600 held-out rows, "R² = 0.479" is
really "R² is somewhere between 0.41 and 0.53". Quoting four decimal places implies
a precision you do not have.

**The point estimate (0.479) is well below the cross-validated mean (0.606).** This
particular held-out quarter is a hard one. If this had been your only split, you
would have concluded the model is substantially worse than it is - which is §3's
lesson arriving from a different direction.

> Report intervals. A metric without one is a claim without evidence about its own
> reliability.

## §7 - Search, and the last hiding place for optimism

### Grid, random, and halving

`GridSearchCV` tries every combination. That is fine for two hyperparameters and
ruinous for six: five values each is 15,625 fits.

`RandomizedSearchCV` samples combinations at random for a fixed budget. It is
usually *better*, not merely cheaper, and the reason is worth knowing: in most
problems only a few hyperparameters matter much, and a grid wastes its budget
re-testing the irrelevant ones at many values of the important ones
([Bergstra & Bengio, 2012](https://jmlr.org/papers/v13/bergstra12a.html)).

`HalvingRandomSearchCV` allocates a small budget to many candidates, discards the
worst, and re-invests in the survivors.

### The subtler problem: selection optimism

Suppose you try 200 hyperparameter configurations and report the best cross-validated
score. That number is biased upward - you took a maximum over 200 noisy estimates,
and the maximum of noisy things is optimistic even when nothing is truly better.

**Nested cross-validation** measures the honest performance of the whole
*procedure*, tuning included: an inner loop chooses hyperparameters, an outer loop
evaluates the tuned model on data the inner loop never saw.

In [ ]:
# Why tuning needs two levels of cross-validation. The grid is small on purpose:
# six values are enough to make the optimism visible.
grid = {"model__alpha": [0.01, 0.1, 1, 10, 100, 1000]}
base = Pipeline([("prep", make_preprocessor(NUMERIC_SHORT, CATEGORICAL)),
                 ("model", Ridge())])

# The common mistake: search, then report the best score the search found.
flat = GridSearchCV(base, grid, cv=GroupKFold(4), scoring="r2")
flat.fit(short[NUMERIC_SHORT + CATEGORICAL], y_short, groups=groups_short)

# The correct version: the entire search becomes the estimator, and an outer loop
# scores it on folds the search never saw. The inner loop picks alpha, the outer
# loop measures how well "fit a model and tune alpha" generalises.
nested = cross_val_score(
    GridSearchCV(base, grid, cv=GroupKFold(4), scoring="r2"),
    short[NUMERIC_SHORT + CATEGORICAL], y_short,
    cv=GroupKFold(5), groups=groups_short, scoring="r2",
    # params={"groups": ...} forwards the grouping down into the inner search too,
    # so grouping is respected at both levels rather than only the outer one.
    params={"groups": groups_short})

print(f"best inner CV score, reported as the result : {flat.best_score_:.4f}  "
      f"(α = {flat.best_params_['model__alpha']})")
print(f"nested CV - honest estimate of the procedure: {nested.mean():.4f} "
      f"+/- {nested.std():.4f}")
# The optimism is the price of having chosen the winner by looking at the same
# folds you then report. It is small here, and it grows with the size of the grid.
print(f"optimism from reporting the inner score     : "
      f"{flat.best_score_ - nested.mean():+.4f}")

**+0.018 of optimism**, from choosing among only six candidates. The effect grows
with the size of the search: try 200 configurations and the bias is far larger.

Practical guidance, since nested CV is expensive:

- Tuning a handful of values and reporting the inner score is usually acceptable -
  just say that is what you did.
- Comparing *model families* after tuning each, or searching hundreds of
  configurations, needs nested CV or a separate held-out set.
- Your sealed test set exists precisely so that you have one honest number at the
  end regardless.

## §8 - Common mistakes

| Mistake | Why it is tempting | What to do instead |
|---|---|---|
| Reporting one split's score | it is one number and looks clean | 30 splits ranged 0.726–0.843 |
| `KFold` on grouped data | it is the default | +0.045 R², +0.041 AUC of inflation here |
| Trusting a narrow interval | precision looks like quality | the broken method had *a quarter* of the spread |
| "0.606 > 0.591, so Ridge wins" | the numbers are ordered | paired p = 0.531; signs disagree across folds |
| `ttest_ind` on CV scores | it is the test people remember | the folds are paired; use `ttest_rel` |
| Reporting the best tuned CV score | it is the result of your tuning | it is a maximum over noisy estimates; +0.018 optimism at six candidates |
| Treating a feature as safe because it passes the deployment test | the test is a good one | `minimum_nights` passed it and still invalidated the target |

## §9 - Reflection

1. Rewrite, in two sentences for Client B, what your model now does and does not
   cover after the re-scoping.
2. We discarded 4,627 long-stay listings. Design the analysis you would run to serve
   that segment, in three bullet points.
3. A colleague reports "R² = 0.83" on this dataset. Write the three questions you
   would ask, in the order you would ask them.

## §10 - Knowledge check

1. Thirty single splits of one model gave R² from 0.726 to 0.843. What does that
   imply about any single reported held-out score?
2. Random KFold gave a *higher* mean and a *smaller* standard deviation than
   GroupKFold. Explain both.
3. `minimum_nights` is known at prediction time, so it passes the deployment test.
   Why did we remove it anyway?
4. Two models score 0.606 ± 0.100 and 0.591 ± 0.069 on the same folds. What test do
   you run, and why does pairing matter?
5. What does nested cross-validation estimate that a tuned inner-loop score does not?

## Summary

- **A single held-out score is a sample of size one.** Thirty splits of one model
  ranged 0.726–0.843.
- The splitter must match the data's structure. Random KFold inflated R² by
  **+0.046** and AUC by **+0.041** - while reporting a standard deviation four times
  narrower (0.003 vs 0.013). It looked like the careful analysis.
- **A narrow interval is not evidence of correctness.** Folds that all make the same
  mistake agree beautifully.
- `minimum_nights` had permutation importance **0.646**, five times the next
  feature, because `price` is the per-night rate for a stay of length
  `minimum_nights` (corr **0.98**). The target did not mean one thing.
- **Leakage is a relationship between a column, a target and a decision** - not a
  property of a column. The same feature is valid for a revenue analyst and
  invalid for Client B.
- Re-scoping cost us 42% of the data and took R² from 0.834 to **0.606**. The full
  descent is **0.999 → 0.973 → 0.957 → 0.834 → 0.606**, and every step was an
  improvement.
- Ridge 0.606 vs HistGB 0.591 is **not a difference**: paired p = 0.531, and the
  per-fold signs disagree. Contrast Session 5, where p = 0.0017 with all five signs
  agreeing.
- Reporting a tuned inner-loop score carried **+0.018** of optimism over six
  candidates. Disclose which number you are reporting.

## Key takeaways

1. Never report a metric without a spread.
2. Compare models fold by fold, and look at the signs.
3. Ask what the target *means* before asking how well you predict it.

## Further exploration

**Essential**
- scikit-learn user guide, *Cross-validation*:
  https://scikit-learn.org/stable/modules/cross_validation.html
- Bergstra & Bengio (2012), *Random Search for Hyper-Parameter Optimization*, JMLR
  13. https://jmlr.org/papers/v13/bergstra12a.html

**Recommended**
- Cawley & Talbot (2010), *On Over-fitting in Model Selection and Subsequent
  Selection Bias in Performance Evaluation*, JMLR 11 - §7's optimism, measured
  carefully. https://jmlr.org/papers/v11/cawley10a.html
- Varoquaux (2018), *Cross-validation failure: small sample sizes lead to large
  error bars*, NeuroImage - how wide these intervals really are.

**Advanced**
- Dietterich (1998), *Approximate Statistical Tests for Comparing Supervised
  Classification Learning Algorithms*, Neural Computation 10(7) - why the naive
  paired t-test on CV folds is itself imperfect, since the folds share training
  data. Our conclusion here (no difference) is safe; be careful using the same test
  to claim a *small* significant difference.

---

**Next session:** ensembles. Boosting will beat everything on the classification
task and lose to Ridge on this one - and you now have the tools to say which of
those two statements is a real finding.